# NB06 — Functional enrichment and GSEA
* **GSEA** (`fgsea` multilevel) on ranked statistics: meta-analysis z (log2FC / SE) and each cohort's moderated t.
* **Over-representation** (`fgsea::fora`) of meta-significant genes, up and down separately, with the shared
  meta-analysis gene universe as background.
* Collections: Hallmark, GO BP / CC / MF, Reactome (MSigDB via `msigdbr`) and **SynGO** terms (from your SynGO files).
  KEGG is left out because of licensing.
* Redundant GO BP terms are collapsed (`fgsea::collapsePathways`, top 300 significant terms) for reading.
* Pathways are tagged by keyword as *established GBM theme* or *possible cell-composition theme*. Tags guide reading;
  they are not findings.

Multiple testing: BH within each collection and ranking. Size limits come from `config.json`.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
EN = CFG["enrichment"]; M = CFG["meta"]
ENR = P("enrich")
print("SMOKE =", SMOKE)

In [ ]:
meta_path = P("meta_analysis", "meta_analysis_full.csv")
require_files([meta_path, P("enrich", "msigdb_long.csv.gz")], "Run NB05 first.")
meta = pd.read_csv(meta_path)

# gene sets: MSigDB + SynGO terms
gsets = pd.read_csv(P("enrich", "msigdb_long.csv.gz"))
syn_terms = Path(str(ROOT)) / "03_Processed_Datasets" / "SynGO" / "syngo_term_membership.csv"
if syn_terms.exists():
    st = pd.read_csv(syn_terms)
    names = {}
    onto = P("raw") / "SynGO" / "ontologies.xlsx"
    if onto.exists():
        o = pd.read_excel(onto, engine="openpyxl")
        ncol = find_col(o, ["name", "term_name", "go_name"], required=False)
        if ncol:
            names = dict(zip(o["id"].astype(str), o[ncol].astype(str)))
    st["gs_name"] = "SYNGO_" + st.go_id.astype(str).map(lambda g: (names.get(g, g)).upper().replace(" ", "_")[:80] + "_" + g.replace(":", ""))
    syn = pd.DataFrame({"collection": "SYNGO", "gs_name": st.gs_name, "gene_symbol": st.hgnc_symbol})
    gsets = pd.concat([gsets, syn], ignore_index=True)
    print("SynGO terms added:", syn.gs_name.nunique())
else:
    log("SynGO term file not found; SynGO enrichment skipped.", "WARN")
sets_for_r = P("enrich", "genesets_for_fgsea.csv.gz"); gsets.drop_duplicates().to_csv(sets_for_r, index=False)

rank_dir = P("enrich", "rank_files"); rank_dir.mkdir(exist_ok=True)
for f in rank_dir.glob("*.csv"):
    f.unlink()
pd.DataFrame({"gene": meta.gene, "stat": meta.meta_log2FC / meta.meta_SE}).to_csv(rank_dir / "META.csv", index=False)
for f in sorted(P("de").glob("*_DE_full.csv")):
    d = pd.read_csv(f).dropna(subset=["t"]).drop_duplicates("gene")
    pd.DataFrame({"gene": d.gene, "stat": d.t}).to_csv(rank_dir / f"{f.name.replace('_DE_full.csv', '')}.csv", index=False)
sig_up = meta[(meta.meta_FDR < M["fdr"]) & (meta.meta_log2FC > 0)].gene
sig_dn = meta[(meta.meta_FDR < M["fdr"]) & (meta.meta_log2FC < 0)].gene
pd.DataFrame({"gene": sig_up}).to_csv(ENR / "ora_up_genes.csv", index=False)
pd.DataFrame({"gene": sig_dn}).to_csv(ENR / "ora_down_genes.csv", index=False)
pd.DataFrame({"gene": meta.gene}).to_csv(ENR / "ora_universe.csv", index=False)
print(f"ORA inputs: up={len(sig_up)}, down={len(sig_dn)}, universe={len(meta)}; rank files: {[p.stem for p in rank_dir.glob('*.csv')]}")

In [ ]:
R_ENR = r'''
suppressMessages({library(fgsea); library(data.table)})
args <- commandArgs(TRUE)
sets_f <- args[1]; rank_dir <- args[2]; out_dir <- args[3]
minS <- as.integer(args[4]); maxS <- as.integer(args[5]); enr_dir <- args[6]
set.seed(1)
sets <- fread(sets_f)
colls <- unique(sets$collection)
pw_by_coll <- lapply(colls, function(cl) { s <- sets[collection == cl]; lapply(split(s$gene_symbol, s$gs_name), unique) })
names(pw_by_coll) <- colls
ru <- fread(file.path(enr_dir, "ora_universe.csv"))$gene
up <- fread(file.path(enr_dir, "ora_up_genes.csv"))$gene
dn <- fread(file.path(enr_dir, "ora_down_genes.csv"))$gene
flat <- function(x) vapply(x, function(v) paste(head(v, 40), collapse = ";"), "")
for (rf in list.files(rank_dir, pattern = "\\.csv$", full.names = TRUE)) {
  nm <- sub("\\.csv$", "", basename(rf))
  rk <- fread(rf); st <- setNames(rk$stat, rk$gene); st <- st[is.finite(st)]; st <- st[!duplicated(names(st))]
  for (cl in colls) {
    pw <- pw_by_coll[[cl]]
    res <- suppressWarnings(fgseaMultilevel(pw, st, minSize = minS, maxSize = maxS, nproc = 2))
    res <- res[order(pval)]
    res[, collapsed_main := NA]
    sig <- res[padj < 0.05]
    if (cl == "GO:BP" && nm == "META" && nrow(sig) > 1) {
      sig <- head(sig, 300)
      col <- suppressWarnings(collapsePathways(sig, pw, st))
      res[, collapsed_main := pathway %in% col$mainPathways]
    }
    res[, leadingEdge := flat(leadingEdge)]
    fwrite(res, file.path(out_dir, paste0("GSEA_", nm, "_", gsub(":", "", cl), ".csv")))
  }
  cat("GSEA done:", nm, "\n")
}
for (cl in colls) {
  pw <- pw_by_coll[[cl]]
  for (dir in c("up", "down")) {
    g <- if (dir == "up") up else dn
    if (length(g) < 5) next
    o <- fora(pw, g, ru, minSize = minS, maxSize = maxS)
    o[, overlapGenes := flat(overlapGenes)]
    fwrite(o[order(pval)], file.path(out_dir, paste0("ORA_", dir, "_", gsub(":", "", cl), ".csv")))
  }
}
cat("enrichment finished\n")
'''
print(run_r(write_r("fgsea_run.R", R_ENR), [sets_for_r, rank_dir, ENR, EN["min_size"], EN["max_size"], ENR], timeout=10800))

In [ ]:
# ---- tag pathways and compare cohorts ------------------------------------------------------------------
ESTABLISHED = ["CELL_CYCLE", "MITOTIC", "E2F", "G2M", "DNA_REPLICATION", "EXTRACELLULAR_MATRIX", "COLLAGEN", "ANGIOGENESIS",
               "HYPOXIA", "EPITHELIAL_MESENCHYMAL", "INFLAMMATORY", "INTERFERON", "COMPLEMENT", "IMMUNE", "LEUKOCYTE", "TNFA", "NFKB"]
COMPOSITION = ["SYNAP", "NEURO", "AXON", "DENDRIT", "MYELIN", "GLIA", "OLIGODENDRO", "ASTROCYTE", "NERVE", "ENSHEATH", "SYNGO_"]


def tag(name):
    u = str(name).upper()
    if any(k in u for k in COMPOSITION):
        return "possible_composition_theme"
    if any(k in u for k in ESTABLISHED):
        return "established_GBM_theme"
    return "other"


tables = []
for f in sorted(ENR.glob("GSEA_META_*.csv")):
    t = pd.read_csv(f); t["tag"] = t.pathway.map(tag); t["collection"] = f.stem.replace("GSEA_META_", "")
    tables.append(t)
require(bool(tables), "No GSEA output found; check the R log above.")
allg = pd.concat(tables)
allg.to_csv(ENR / "GSEA_META_all_collections_tagged.csv", index=False)
sigt = allg[allg.padj < EN["fdr"]]
print("Significant GSEA terms (meta ranking) by collection and tag:")
print(sigt.groupby(["collection", "tag"]).size().unstack(fill_value=0).to_string())
print("\nTop 10 by |NES| (meta, Hallmark):")
h = allg[(allg.collection == "H") & (allg.padj < EN["fdr"])].sort_values("NES", key=abs, ascending=False)
print(h.head(10)[["pathway", "NES", "padj", "tag"]].round(4).to_string(index=False))

cmp_rows = []
for coll in ["H", "GOBP", "SYNGO"]:
    parts = {}
    for f in sorted(ENR.glob(f"GSEA_*_{coll}.csv")):
        nm = f.stem[len("GSEA_"):-len(f"_{coll}")]
        parts[nm] = pd.read_csv(f).set_index("pathway")["NES"]
    if len(parts) >= 2:
        d = pd.DataFrame(parts).dropna()
        cohort_cols = [c for c in d.columns if c != "META"]
        d["same_sign_in_all_cohorts"] = np.sign(d[cohort_cols]).nunique(axis=1) == 1
        d["collection"] = coll; d["tag"] = [tag(i) for i in d.index]
        cmp_rows.append(d.reset_index().rename(columns={"index": "pathway"}))
if cmp_rows:
    cmp_df = pd.concat(cmp_rows); cmp_df.to_csv(ENR / "GSEA_NES_cohort_comparison.csv", index=False)
    print(cmp_df.groupby(["collection", "same_sign_in_all_cohorts"]).size().to_string())

In [ ]:
# ---- ORA figure (GO BP, meta; up and down) ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, d, colr in zip(axes, ["up", "down"], ["#1f3a93", "#0f9d94"]):
    f = ENR / f"ORA_{d}_GOBP.csv"
    if not f.exists():
        ax.set_title(f"{d}: no ORA output (too few genes)"); continue
    t = pd.read_csv(f).head(12).iloc[::-1]
    ax.barh(t.pathway.str.replace("GOBP_", "").str.lower().str.replace("_", " ").str.slice(0, 55), -np.log10(t.padj.clip(lower=1e-300)), color=colr)
    ax.set_xlabel("-log10 adjusted p"); ax.set_title(f"GO BP over-representation: {d} in GBM", fontsize=10)
fig.tight_layout(); savefig(fig, "ORA_GOBP_meta", folder="enrich"); plt.close(fig)
manifest_update("enrichment", "settings", {"fdr": EN["fdr"], "min_size": EN["min_size"], "max_size": EN["max_size"],
                                           "background": "shared meta-analysis gene universe",
                                           "collections": sorted(gsets.collection.unique().tolist())})
print("\nNB06 complete. Next: NB07 (co-expression, optional) or NB10 (single cell).")